## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-25-03-54-43 +0000,bbc,"Kilmar Ábrego García leaving US for Spain, end...",https://www.bbc.co.uk/news/articles/c6808pyy5d...
1,2026-11-25-03-52-08 +0000,nyt,"Martha Firestone Ford, Owner of Detroit Lions,...",https://www.nytimes.com/2026/11/24/business/ma...
2,2026-11-25-03-51-55 +0000,nyt,"Abrego Garcia Will Voluntarily Leave U.S., End...",https://www.nytimes.com/2026/11/24/us/politics...
3,2026-11-25-03-44-07 +0000,bbc,US attorney general dodges questions about Nvi...,https://www.bbc.co.uk/news/articles/crpdw1qx7j...
4,2026-11-25-03-29-08 +0000,nypost,Heart-pounding moment killer whales converge o...,https://nypost.com/2026/11/24/us-news/video-sh...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2491/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
27,nvidia,41
6,trump,40
290,huang,30
289,jensen,24
59,up,14


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
401,2026-11-24-08-52-18 +0000,nypost,Trump shrugs off GOP concerns that Jensen Huan...,https://nypost.com/2026/11/24/us-news/trump-sh...,187
232,2026-11-24-18-28-12 +0000,nypost,Trump says cheating Nvidia CEO Jensen Huang an...,https://nypost.com/2026/11/24/us-news/trump-sa...,168
75,2026-11-25-00-18-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...,168
365,2026-11-24-10-53-37 +0000,bbc,Trump faces backlash after saying he 'convince...,https://www.bbc.co.uk/news/articles/cq5y1k8ez4...,163
396,2026-11-24-09-00-00 +0000,cbc,Why Trump's move to get Jensen Huang reinstate...,https://www.cbc.ca/news/world/trump-jensen-hua...,160


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
401,187,2026-11-24-08-52-18 +0000,nypost,Trump shrugs off GOP concerns that Jensen Huan...,https://nypost.com/2026/11/24/us-news/trump-sh...
75,73,2026-11-25-00-18-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...
14,56,2026-11-25-03-00-00 +0000,wsj,A Democratic push to enact Medicare for All is...,https://www.wsj.com/politics/policy/democrats-...
83,50,2026-11-24-23-49-03 +0000,nypost,Hudson River Trading expanding at 3 World Tra...,https://nypost.com/2026/11/24/business/hudson-...
30,49,2026-11-25-02-00-00 +0000,wsj,What would happen to the U.S. economy if the A...,https://www.wsj.com/economy/worse-than-the-dot...
206,47,2026-11-24-20-06-02 +0000,latimes,A man's severed head was found in a rural Cali...,https://www.latimes.com/california/story/2026-...
414,46,2026-11-24-04-15-00 +0000,wsj,A federal appeals court has cleared the way fo...,https://www.wsj.com/politics/national-security...
7,42,2026-11-25-03-18-52 +0000,nypost,Elderly Alzheimer-stricken pianist crosses 2 s...,https://nypost.com/2026/11/24/us-news/elderly-...
70,42,2026-11-25-00-30-48 +0000,nypost,Trump wishes Robert Pattinson ‘luck’ as new Ja...,https://nypost.com/2026/11/24/us-news/trump-wi...
77,42,2026-11-25-00-13-52 +0000,nypost,"Andrew Giuliani’s daughter, 4, steals spotligh...",https://nypost.com/2026/11/24/us-news/andrew-g...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
